In [ ]:
import pandas as pd
import wandb
wandb.login()

from tqdm import tqdm
import pandas as pd
import wandb

def extract_complete_data(runs):
    data = []
    for run in tqdm(runs, desc='Processing runs'):
        config = dict(run.config)
        history = run.history(keys=["environment", "_step"])

        rewards_list = []

        if 'environment' in history.columns:
            for index, row in tqdm(history.iterrows(), total=history.shape[0], leave=False, desc='Gathering environment data'):
                if not pd.isna(row['environment']):
                    rewards_list.append(row['environment'])

        if rewards_list:
            data_point = {
                'name': run.name,
                'rewards_list': rewards_list,
                **config
            }
            data.append(data_point)

    df = pd.DataFrame(data)

    cols = ['name', 'rewards_list'] + [col for col in df.columns if col not in {'name', 'rewards_list'}]
    df = df[cols]
    return df




wandb.login()
api = wandb.Api()
project_name = "WB_CLTV"
runs = api.runs(path=project_name)

df = extract_complete_data(runs)


In [ ]:
df.head()

In [ ]:
df.dataset_types = df.dataset_types.astype("str")
df.rename(columns={'_runtime': 'runtime'}, inplace=True)


datasets_t = {
    "['random-v2', 'expert-v2']": "random-expert", 
    "['random-v2', 'medium-v2']": "random-medium", 
    "['medium-v2', 'expert-v2']": "medium-expert"
}
df['dataset_types'] = df['dataset_types'].map(datasets_t)

domains = ["Ant", "HalfCheetah", "Hopper", "Walker2d"]
tt = SortedDict(list(zip(list(range(0, 12, 3)), domains)))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem, t
from sortedcontainers import SortedDict

methods = ["CQL", "IQL"]


def prepare_and_aggregate_data(df):
    df = df.explode('rewards_list')
    df['rewards_list'] = pd.to_numeric(df['rewards_list'])
    df['time_step'] = df.groupby(['env', 'dataset_types', 'method', 'baseline', 'seed']).cumcount() + 1

    aggregated_data = df.groupby(['env', 'dataset_types', 'method', 'baseline', 'time_step']).agg(
        mean_reward=('rewards_list', 'mean'),
        std_dev=('rewards_list', 'std'),
        count=('rewards_list', 'size')
    ).reset_index()

    aggregated_data['95% CI'] = aggregated_data.apply(
        lambda x: (x['std_dev'] / np.sqrt(x['count']) * t.ppf((1 + 0.95) / 2., x['count'] - 1)) if x['count'] > 1 else 0,
        axis=1
    )

    aggregated_data['std_dev'] = aggregated_data.apply(
        lambda x: 0 if pd.isna(x['std_dev']) or x['count'] <= 1 else x['std_dev'],
        axis=1
    )

    return aggregated_data

prepared_data = prepare_and_aggregate_data(df)

In [ ]:
results = {}
for index, row in prepared_data.iterrows():
    key = f"{row['env']}-{row['dataset_types']}-{row['method']}-{row['baseline']}"
    results[key] = (row['mean_reward'], row['std_dev'])


In [ ]:
table_data = {k: f"{v[0]:.2f} ± {v[1]:.2f}" for k, v in results.items()}


df_r = pd.DataFrame(list(table_data.items()), columns=['Env-Dataset-Base Algorithm-Method', 'Normalized Score ± Std'])
print(df_r)